# 03 — İnsan Faktörleri & Yorgunluk Modeli
### Osman-Sheridan Yorgunluk Modeli + THERP Hata Analizi

**Bu notebook'ta:**
- Kümülatif yorgunluk modelinin matematiksel türetimi
- Yerkes-Dodson ters-U arousal-performans eğrisi
- THERP hata tipi dağılımı
- 18 kişilik ekipte bireysel farklılıklar
- Yorgunluk → hata olasılığı → pit stop süresi zinciri

---


## 1. Yorgunluk Modeli — Matematik

### Osman-Sheridan Kümülatif Yorgunluk:

$$F(n) = F_{max} \cdot \left(1 - e^{-\beta_{eff} \cdot W_n}\right)$$

Fitness seviyesi biriktirmeyi yavaşlatır:
$$\beta_{eff} = \frac{\beta}{1 + fitness}$$

### Hata Olasılığı (THERP):

$$P_{error}(F) = P_{base} \cdot e^{\gamma \cdot F} \cdot \frac{1}{1 + k_A \cdot \text{Bell}(A)}$$

Bell fonksiyonu (Yerkes-Dodson):
$$\text{Bell}(A) = e^{-\frac{1}{2}\left(\frac{A - A_{opt}}{\sigma_A}\right)^2}$$


In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pitstop.simulation.human_factors import CrewMember, PitCrew, simulate_race_pitstops

# Yorgunluk eğrisi — farklı fitness seviyeleri
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('İnsan Faktörleri Modeli', fontsize=13, fontweight='bold')

W = np.linspace(0, 20, 300)  # kümülatif workload
beta = 0.03

ax = axes[0, 0]
for fitness, color, label in [(0.2, '#E74C3C', 'Düşük fitness'), (0.5, '#F39C12', 'Orta fitness'),
                               (0.8, '#27AE60', 'Yüksek fitness'), (0.95, '#2980B9', 'Elite fitness')]:
    beta_eff = beta / (1 + fitness)
    F = 1.0 * (1 - np.exp(-beta_eff * W))
    ax.plot(W, F, color=color, linewidth=2.5, label=label)
ax.set_xlabel('Kümülatif Workload (n pit stops)')
ax.set_ylabel('Yorgunluk F ∈ [0,1]')
ax.set_title('Yorgunluk Birikimi — Fitness Etkisi')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)

# Hata olasılığı vs yorgunluk
ax = axes[0, 1]
F_vals = np.linspace(0, 1, 300)
gamma = 2.5
for p_base, color, label in [(0.010, '#2980B9', 'Jack (p_base=1.0%)'),
                              (0.018, '#27AE60', 'Gunman (p_base=1.8%)'),
                              (0.025, '#E74C3C', 'Wing crew (p_base=2.5%)')]:
    p_err = p_base * np.exp(gamma * F_vals)
    ax.plot(F_vals, p_err * 100, color=color, linewidth=2.5, label=label)
ax.set_xlabel('Yorgunluk seviyesi F')
ax.set_ylabel('Hata olasılığı (%)')
ax.set_title('P_error(F) = P_base · exp(γ·F)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 25)

# Yerkes-Dodson eğrisi
ax = axes[1, 0]
A = np.linspace(0, 1, 300)
A_opt, sigma_A, k_A = 0.65, 0.25, 0.20
bell = np.exp(-0.5 * ((A - A_opt) / sigma_A) ** 2)
performance_mod = 1 + k_A * bell
ax.plot(A, performance_mod, 'steelblue', linewidth=3)
ax.axvline(A_opt, color='red', linestyle='--', linewidth=1.5, label=f'Optimal arousal={A_opt}')
ax.fill_between(A, 1, performance_mod, alpha=0.2, color='green')
ax.set_xlabel('Arousal seviyesi A ∈ [0,1]')
ax.set_ylabel('Performans çarpanı')
ax.set_title('Yerkes-Dodson Ters-U Eğrisi')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0.95, 1.25)

# 5 pit stop boyunca yorgunluk artışı
ax = axes[1, 1]
df = simulate_race_pitstops(n_stops=8, skill_level=0.80, seed=42)
ax.plot(df['stop_number'], df['avg_fatigue'], 'steelblue', linewidth=2.5,
        marker='o', label='Ortalama yorgunluk')
ax.plot(df['stop_number'], df['max_fatigue'], 'tomato', linewidth=2.5,
        marker='s', linestyle='--', label='Max yorgunluk')
ax2_twin = ax.twinx()
ax2_twin.plot(df['stop_number'], df['pit_time'], 'green', linewidth=2, marker='^',
              label='Pit süresi')
ax.set_xlabel('Pit stop sayısı')
ax.set_ylabel('Yorgunluk', color='navy')
ax2_twin.set_ylabel('Pit stop süresi (s)', color='green')
ax.set_title('Yarış Boyunca Yorgunluk Birikimi')
ax.legend(loc='upper left', fontsize=9)
ax2_twin.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('03_human_factors.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. 18 Kişilik Ekip — Bireysel Varyasyon

In [ ]:
import pandas as pd
from pitstop.simulation.human_factors import PitCrew
import numpy as np

# 5 pit stop simüle et, ardından bireysel rapor çıkar
np.random.seed(0)
crew = PitCrew.create_standard(skill_level=0.80)
rng = np.random.default_rng(0)
for stop in range(5):
    crew.simulate_pit_stop(rng)

report = crew.fatigue_report()
print("=== 18-Kişi Ekip Yorgunluk Raporu (5 pit stop sonrası) ===")
print(report.sort_values('error_prob_%', ascending=False).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.barh(report['role'], report['fatigue'], color=['#E74C3C' if f > 0.08 else '#F39C12' if f > 0.05 else '#27AE60'
                                                    for f in report['fatigue']])
ax.set_xlabel('Yorgunluk seviyesi')
ax.set_title('Bireysel Yorgunluk — 18 Kişi (5 stop sonrası)', fontweight='bold')
ax.axvline(0.05, color='orange', linestyle='--', linewidth=1.5, label='Orta risk')
ax.axvline(0.08, color='red', linestyle='--', linewidth=1.5, label='Yüksek risk')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)

ax2 = axes[1]
ax2.barh(report['role'], report['error_prob_%'],
         color=['#E74C3C' if e > 2.0 else '#F39C12' if e > 1.0 else '#27AE60'
                for e in report['error_prob_%']])
ax2.set_xlabel('Hata olasılığı (%)')
ax2.set_title('Bireysel Hata Riski', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('03_individual_fatigue.png', dpi=150, bbox_inches='tight')
plt.show()
